<a href="https://colab.research.google.com/github/shahaanabbas-13/Quantum-Diabetes-Prediction/blob/main/notebooks/03_Classical_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Classical ML baselines
Compare four classical classifiers using training-only cross-validation. The holdout set is reserved for final evaluation.

In [2]:
%pip -q install scikit-learn joblib

In [6]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate


# Separate features and target
X = df.drop(columns=TARGET).copy()
y = df[TARGET].astype(int).copy()

# Replace invalid zero values with NaN
X[INVALID_ZERO_COLS] = X[INVALID_ZERO_COLS].replace(0, np.nan)

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training outcome distribution:")
print(y_train.value_counts(normalize=True).round(3))


models={
 "Logistic Regression": LogisticRegression(max_iter=2000,class_weight="balanced"),
 "Random Forest": RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight="balanced",random_state=42),
 "RBF-SVM": SVC(kernel="rbf",C=1.0,gamma="scale",class_weight="balanced",probability=True,random_state=42),
 "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,model in models.items():
    steps=[("imputer",SimpleImputer(strategy="median"))]
    if name in ["Logistic Regression","RBF-SVM"]: steps.append(("scaler",StandardScaler()))
    pipe=Pipeline(steps+[("model",model)])

    scores = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=-1
    )

    rows.append({"model":name,**{f"{m}_mean":scores[f"test_{m}"].mean() for m in ["accuracy","recall","f1","roc_auc"]},
                 **{f"{m}_std":scores[f"test_{m}"].std() for m in ["accuracy","recall","f1","roc_auc"]}})

results=pd.DataFrame(rows)
display(results.sort_values("recall_mean",ascending=False))

# Create results directory if it doesn't exist
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save the experimental results
RESULTS_PATH = RESULTS_DIR / "classical_cv.csv"

results.to_csv(RESULTS_PATH, index=False)

print("Results saved successfully!")
print("Location:", RESULTS_PATH)

# Verify the saved file
print("\nSaved results:")
display(pd.read_csv(RESULTS_PATH))

Training data: (614, 8)
Testing data: (154, 8)
Training outcome distribution:
Outcome
0    0.651
1    0.349
Name: proportion, dtype: float64


,model,accuracy_mean,recall_mean,f1_mean,roc_auc_mean,accuracy_std,recall_std,f1_std,roc_auc_std
2,RBF-SVM,0.760536,0.761683,0.690163,0.837141,0.033953,0.051741,0.028713,0.030934
0,Logistic Regression,0.762202,0.700997,0.672385,0.844381,0.011053,0.039478,0.013319,0.016074
1,Random Forest,0.771971,0.630786,0.659681,0.832183,0.028285,0.010751,0.027859,0.023682
3,Gradient Boosting,0.755684,0.598117,0.630750,0.819395,0.018019,0.030798,0.013417,0.016082


Results saved successfully!
Location: /content/results/classical_cv.csv

Saved results:


,model,accuracy_mean,recall_mean,f1_mean,roc_auc_mean,accuracy_std,recall_std,f1_std,roc_auc_std
0,Logistic Regression,0.762202,0.700997,0.672385,0.844381,0.011053,0.039478,0.013319,0.016074
1,Random Forest,0.771971,0.630786,0.659681,0.832183,0.028285,0.010751,0.027859,0.023682
2,RBF-SVM,0.760536,0.761683,0.690163,0.837141,0.033953,0.051741,0.028713,0.030934
3,Gradient Boosting,0.755684,0.598117,0.630750,0.819395,0.018019,0.030798,0.013417,0.016082


## Evaluation note
These are cross-validation estimates. Do not call them test-set results. Final holdout evaluation and threshold selection will be handled separately.

In [7]:
from google.colab import userdata
import requests

token = userdata.get("GITHUB_TOKEN")

response = requests.get(
    "https://api.github.com/user",
    headers={"Authorization": f"Bearer {token}"}
)

print("GitHub connection status:", response.status_code)

if response.status_code == 200:
    print("GitHub authentication successful!")
else:
    print("Please check your token and permissions.")

GitHub connection status: 200
GitHub authentication successful!


In [8]:
import requests
import base64
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

OWNER = "shahaanabbas-13"
REPO = "Quantum-Diabetes-Prediction"
BRANCH = "main"

local_file = "/content/results/classical_cv.csv"
github_file = "results/classical_cv.csv"

with open(local_file, "rb") as f:
    content = base64.b64encode(f.read()).decode("utf-8")

url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{github_file}"

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}

payload = {
    "message": "Add classical ML experiment results",
    "content": content,
    "branch": BRANCH
}

response = requests.put(url, headers=headers, json=payload)

print("Status:", response.status_code)
print(response.json().get("message", "Upload completed"))

Status: 201
Upload completed
